# Embeddings and Vector Database Operations

This notebook demonstrates the complete workflow for generating embeddings and storing them in a vector database for semantic search.

## What You'll Learn
1. Generate embeddings with Ollama
2. Store vectors in OpenSearch
3. Perform similarity search
4. Understand embedding models and their trade-offs
5. Learn vector search concepts (similarity metrics, algorithms)
6. Learn VectorDB operator concepts
7. Best practices for embeddings and vector databases

## Prerequisites
- **Ollama** running on `http://localhost:11434`
- **Model** `nomic-embed-text` pulled: `ollama pull nomic-embed-text`
- **OpenSearch** running on `https://localhost:9200` (optional, for vector storage; set `OPENSEARCH_USE_SSL=false` if using plain HTTP)
- Sample documents available

## Setup and Imports

In [ ]:
import sys
from pathlib import Path
from pprint import pprint
import pandas as pd
import numpy as np

# Add src to path if needed
import os
if 'PYTHONPATH' not in os.environ:
    src_path = Path.cwd().parent.parent / "src"
    sys.path.insert(0, str(src_path))

from docpipe.lib.docpipe_flow_manager import DocpipeFlowManager

print("✓ Imports loaded successfully")

## Load Environment Variables

In [ ]:
# Set OpenSearch credentials
import os
from pathlib import Path

# Option 1: Load from .env file (recommended)
# Copy .env.example to .env in the project root and update with your credentials
# Look for .env in project root (two levels up from this notebook)
project_root = Path.cwd()
if (project_root / 'examples' / 'notebooks').exists():
    # We're already in project root
    env_file = project_root / '.env'
else:
    # We're in a subdirectory, go up to find project root
    env_file = project_root.parent.parent / '.env'

if env_file.exists():
    from dotenv import load_dotenv
    load_dotenv(env_file, override=True)
    print(f"✓ Loaded credentials from .env file: {env_file}")
else:
    print(f"⚠ .env file not found at: {env_file}")
    print("  Tip: Copy .env.example to .env in project root and update OPENSEARCH_USERNAME and OPENSEARCH_PASSWORD")

# Option 2: Or set them in your shell before starting Jupyter:
# export OPENSEARCH_USERNAME=your_username
# export OPENSEARCH_PASSWORD=your_password

# Option 3: Set environment variables manually in notebook (if .env not available)
if not os.getenv('OPENSEARCH_USERNAME'):
    os.environ['OPENSEARCH_USERNAME'] = 'admin'
if not os.getenv('OPENSEARCH_PASSWORD'):
    os.environ['OPENSEARCH_PASSWORD'] = '<YOUR-OPENSEARCH-PASSWORD>'

# Set OPENSEARCH_USE_SSL=false in your environment if your OpenSearch uses plain HTTP
OPENSEARCH_USE_SSL = os.getenv('OPENSEARCH_USE_SSL', 'true').lower() != 'false'
OPENSEARCH_SCHEME = 'https' if OPENSEARCH_USE_SSL else 'http'
OPENSEARCH_BASE_URL = f"{OPENSEARCH_SCHEME}://localhost:9200"

print(f"✓ OpenSearch credentials configured (username: {os.environ.get('OPENSEARCH_USERNAME', 'admin')})")
print(f"✓ OpenSearch URL: {OPENSEARCH_BASE_URL}")

## Check Prerequisites

In [ ]:
import requests

def check_services():
    """Check if required services are running"""
    services_ok = True
    
    # Check Ollama
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=2)
        if response.status_code == 200:
            models = response.json().get('models', [])
            model_names = [m['name'] for m in models]
            if any('nomic-embed-text' in name for name in model_names):
                print("✓ Ollama is running with nomic-embed-text model")
            else:
                print("✗ Ollama running but nomic-embed-text not found")
                print("  Run: ollama pull nomic-embed-text")
                services_ok = False
    except Exception as e:
        print(f"✗ Ollama not running: {e}")
        print("  Start: ollama serve")
        services_ok = False
    
    # Check OpenSearch (optional)
    try:
        import urllib3
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        
        response = requests.get(
            OPENSEARCH_BASE_URL,
            auth=(os.getenv('OPENSEARCH_USERNAME', 'admin'), os.getenv('OPENSEARCH_PASSWORD', 'admin')),
            verify=False,
            timeout=5
        )
        if response.status_code == 200:
            cluster_info = response.json()
            version = cluster_info.get('version', {}).get('number', 'unknown')
            print(f"✓ OpenSearch is running: version {version}")
        else:
            print(f"⚠ OpenSearch responded with status {response.status_code}")
    except requests.exceptions.ConnectionError:
        print("ℹ OpenSearch not running (connection refused)")
    except requests.exceptions.Timeout:
        print("ℹ OpenSearch not responding (timeout)")
    except Exception as e:
        print(f"ℹ OpenSearch check failed: {type(e).__name__}: {e}")
    
    return services_ok

if check_services():
    print("\n✓ All required services are ready!")
else:
    print("\n✗ Please fix required services before continuing")

## 1. Generate Embeddings

Create a pipeline that generates embeddings for document chunks:

In [ ]:
embeddings_flow = {
    "flow_name": "embeddings-generation",
    "description": "Generate embeddings for documents",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt",

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "chunk",
            "type": "chunker",
            "depends_on": ["extract"],
            "config": {
                "chunk_type": "simple",
                "chunk_size": 512,
                "chunk_overlap": 50,
                "doc_column": "content"
            }
        },
        {
            "name": "embeddings",
            "type": "embeddings",
            "depends_on": ["chunk"],
            "config": {
                "provider": "litellm",
                "embeddings_column": "embeddings",
                "doc_column": "content",
                "provider_config": {
                    "model_id": "ollama/nomic-embed-text",
                    "api_base": "http://localhost:11434",
                    "api_key": "<YOUR-API-KEY>"
                }
            }
        }
    ]
}

print("Generating embeddings...")
print("Pipeline: Ingest → Extract → Chunk → Embeddings")
print("Model: nomic-embed-text (768 dimensions)")
print()

try:
    manager = DocpipeFlowManager(flow_def=embeddings_flow)
    manager.execute()
    
    print("\n✓ Embeddings generated successfully!")
    print("\nEmbedding Details:")
    print("  - Model: nomic-embed-text")
    print("  - Dimensions: 768")
    print("  - Provider: Ollama (local)")
    print("  - Cost: Free (local processing)")
except Exception as e:
    print(f"\n✗ Embedding generation failed: {e}")
    print("\nMake sure Ollama is running and model is available")

## 2. Store in Vector Database (OpenSearch)

Store embeddings in OpenSearch for similarity search:

**Note:** This section requires OpenSearch to be running. Skip if not available.

In [ ]:
import requests

INDEX_NAME = "notebook-demo-index"

# Delete the index before re-indexing to avoid stale documents from previous runs
try:
    response = requests.delete(
        f"{OPENSEARCH_BASE_URL}/{INDEX_NAME}",
        auth=(os.environ.get("OPENSEARCH_USERNAME", "admin"), os.environ.get("OPENSEARCH_PASSWORD", "admin")),
        verify=False,
        timeout=5,
    )
    if response.status_code == 200:
        print(f"Deleted existing index '{INDEX_NAME}'")
    elif response.status_code == 404:
        print(f"Index '{INDEX_NAME}' does not exist yet, nothing to delete")
    else:
        print(f"Unexpected response deleting index: {response.status_code}")
except Exception as e:
    print(f"Could not connect to OpenSearch (skipping cleanup): {e}")

In [ ]:
vectordb_flow = {
    "flow_name": "vectordb-storage",
    "description": "Store embeddings in OpenSearch",
    "global_config": {
        "doc_column": "content",
        "disable_validation": False,
        "force_ingest": True
    },
    "flow": [
        {
            "name": "ingest",
            "type": "ingest_source",
            "config": {
                "provider": "filesystem",
                "connection_params": {"paths": ["../../sample_documents"]},
                "include_filter": "txt",

            }}
        },
        {
            "name": "extract",
            "type": "extract_operator",
            "depends_on": ["ingest"],
            "config": {
                "text_extraction": {
                    "provider": "docling_library",
                    "doc_column": "content"
                },
                "entity_extraction": {
                    "provider": "none"
                }
            }
        },
        {
            "name": "chunk",
            "type": "chunker",
            "depends_on": ["extract"],
            "config": {
                "chunk_type": "simple",
                "chunk_size": 512,
                "chunk_overlap": 50,
                "doc_column": "content"
            }
        },
        {
            "name": "embeddings",
            "type": "embeddings",
            "depends_on": ["chunk"],
            "config": {
                "provider": "litellm",
                "embeddings_column": "embeddings",
                "doc_column": "content",
                "provider_config": {
                    "model_id": "ollama/nomic-embed-text",
                    "api_base": "http://localhost:11434",
                    "api_key": "<YOUR-API-KEY>"
                }
            }
        },
        {
            "name": "vectordb",
            "type": "vectordb",
            "depends_on": ["embeddings"],
            "config": {
                "provider": "opensearch",
                "index_name": "notebook-demo-index",
                "doc_id_column": "doc_id_hash",
                "embeddings_column": "embeddings",
                "vector_dimension": 768,
                "create_index": True,
                "provider_config": {
                    "host": "localhost",
                    "port": 9200,
                    "username": os.getenv("OPENSEARCH_USERNAME", "admin"),
                    "password": os.getenv("OPENSEARCH_PASSWORD", "admin"),
                    "use_ssl": OPENSEARCH_USE_SSL,
                    "verify_certs": False,
                    "engine": "faiss",
                    "algorithm": "hnsw",
                    "space_type": "l2",
                    "batch_size": 100
                },
                "available_features": {
                    "doc_id_hash": {
                        "name": "Document ID",
                        "available_for_vector_db": True,
                        "mandatory_for_vector_db": True,
                        "type": "string",
                        "is_primary": True
                    },
                    "content": {
                        "name": "Content",
                        "available_for_vector_db": True,
                        "type": "string"
                    },
                    "doc_name": {
                        "name": "Document Name",
                        "available_for_vector_db": True,
                        "type": "string"
                    },
                    "file_path": {
                        "name": "File Path",
                        "available_for_vector_db": True,
                        "type": "string"
                    },
                    "embeddings": {
                        "name": "Embeddings",
                        "available_for_vector_db": True,
                        "mandatory_for_vector_db": True,
                        "type": "vector"
                    },
                    "chunk_id": {
                        "name": "Chunk ID",
                        "available_for_vector_db": True,
                        "type": "string"
                    },
                    "chunk_index": {
                        "name": "Chunk Index",
                        "available_for_vector_db": True,
                        "type": "integer"
                    }
                },
                "feature_mappings": {
                    "doc_id_hash": "pk",
                    "content": "text",
                    "name": "doc_name",
                    "path": "file_path",
                    "embeddings": "vector_embeddings",
                    "chunk_id": "chunk_id",
                    "chunk_index": "chunk_index"
                }
            }
        }
    ]
}

print("Storing embeddings in OpenSearch...")
print("Pipeline: Ingest → Extract → Chunk → Embeddings → VectorDB")
print("Index: notebook-demo-index")
print("Engine: FAISS with HNSW algorithm")
print()

try:
    manager = DocpipeFlowManager(flow_def=vectordb_flow)
    manager.execute()
    
    print("\n✓ Vectors stored in OpenSearch!")
    print("\nVector Database Configuration:")
    print("  - Index: notebook-demo-index")
    print("  - Engine: FAISS (Facebook AI Similarity Search)")
    print("  - Algorithm: HNSW (Hierarchical Navigable Small World)")
    print("  - Distance: L2 (Euclidean)")
    print("\nYou can now perform similarity searches!")
except Exception as e:
    print(f"\n✗ Vector storage failed: {e}")
    print("\nThis is expected if OpenSearch is not running.")
    print("To use vector storage:")
    print("  1. Start OpenSearch: docker-compose -f docker/docker-compose.opensearch.yml up -d")
    print("  2. Verify: curl -k https://localhost:9200")

### Verify Index Contents

Query the OpenSearch index to see what was stored:

In [ ]:
# Query OpenSearch to see what was stored
import requests
import json

def query_opensearch_index(index_name="notebook-demo-index", size=5):
    """Query OpenSearch index and display contents"""
    try:
        # Get index stats
        stats_url = f"{OPENSEARCH_BASE_URL}/{index_name}/_stats"
        stats_response = requests.get(
            stats_url,
            auth=(os.getenv('OPENSEARCH_USERNAME', 'admin'), 
                  os.getenv('OPENSEARCH_PASSWORD', '')),
            verify=False,
            timeout=5
        )
        
        if stats_response.status_code == 200:
            stats = stats_response.json()
            doc_count = stats['_all']['primaries']['docs']['count']
            size_mb = stats['_all']['primaries']['store']['size_in_bytes'] / 1024 / 1024
            print(f"\n📊 Index Statistics:")
            print(f"  - Index: {index_name}")
            print(f"  - Total Documents: {doc_count}")
            print(f"  - Index Size: {size_mb:.2f} MB")
        
        # Get index mapping
        mapping_url = f"{OPENSEARCH_BASE_URL}/{index_name}/_mapping"
        mapping_response = requests.get(
            mapping_url,
            auth=(os.getenv('OPENSEARCH_USERNAME', 'admin'), 
                  os.getenv('OPENSEARCH_PASSWORD', '')),
            verify=False,
            timeout=5
        )
        
        if mapping_response.status_code == 200:
            mapping = mapping_response.json()
            properties = mapping[index_name]['mappings']['properties']
            print(f"\n📋 Index Schema:")
            for field_name, field_config in properties.items():
                field_type = field_config.get('type', 'unknown')
                if field_type == 'knn_vector':
                    dimension = field_config.get('dimension', 'unknown')
                    print(f"  - {field_name}: {field_type} (dimension: {dimension})")
                else:
                    print(f"  - {field_name}: {field_type}")
        
        # Query sample documents
        search_url = f"{OPENSEARCH_BASE_URL}/{index_name}/_search"
        search_body = {
            "size": size,
            "query": {"match_all": {}},
            "_source": {"excludes": ["vector_embeddings"]}  # Exclude large vector field
        }
        
        search_response = requests.post(
            search_url,
            auth=(os.getenv('OPENSEARCH_USERNAME', 'admin'), 
                  os.getenv('OPENSEARCH_PASSWORD', '')),
            headers={'Content-Type': 'application/json'},
            data=json.dumps(search_body),
            verify=False,
            timeout=5
        )
        
        if search_response.status_code == 200:
            results = search_response.json()
            hits = results['hits']['hits']
            
            print(f"\n📄 Sample Documents (showing {len(hits)} of {doc_count}):")
            for i, hit in enumerate(hits, 1):
                source = hit['_source']
                print(f"\n  Document {i}:")
                print(f"    - ID: {hit['_id']}")
                print(f"    - Document Name: {source.get('doc_name', 'N/A')}")
                print(f"    - Chunk Index: {source.get('chunk_index', 'N/A')}")
                content_preview = source.get('text', '')[:100]
                print(f"    - Content Preview: {content_preview}...")
            
            return True
        print(f"\n✗ Search failed: {search_response.status_code}")
        return False
            
    except requests.exceptions.ConnectionError:
        print("\n✗ Cannot connect to OpenSearch. Make sure it's running on localhost:9200")
        return False
    except Exception as e:
        print(f"\n✗ Error querying index: {e}")
        return False

# Query the index
query_opensearch_index()

## 3. Perform Similarity Search

Now let's perform semantic similarity search on the documents we just ingested and stored in OpenSearch:

In [ ]:
def perform_similarity_search(query_text, index_name="notebook-demo-index", k=5):
    """Perform similarity search on the ingested documents using OpenSearch KNN"""
    try:
        # Generate embedding for the query using Ollama
        import requests
        
        print(f"Generating embedding for query: '{query_text}'")
        ollama_url = "http://localhost:11434/api/embeddings"
        ollama_payload = {
            "model": "nomic-embed-text",
            "prompt": query_text
        }
        
        response = requests.post(ollama_url, json=ollama_payload, timeout=30)
        
        if response.status_code != 200:
            print(f"Failed to generate query embedding: {response.status_code}")
            return None
            
        query_embedding = response.json()['embedding']
        print(f"Query embedding generated: {len(query_embedding)} dimensions\n")
        
        # Perform KNN search in OpenSearch on the ingested documents
        search_url = f"{OPENSEARCH_BASE_URL}/{index_name}/_search"
        
        search_body = {
            "size": k,
            "query": {
                "knn": {
                    "vector_embeddings": {
                        "vector": query_embedding,
                        "k": k
                    }
                }
            },
            "_source": ["text", "doc_name", "chunk_index"]
        }
        
        search_response = requests.post(
            search_url,
            auth=(os.getenv('OPENSEARCH_USERNAME', 'admin'), 
                  os.getenv('OPENSEARCH_PASSWORD', '')),
            headers={'Content-Type': 'application/json'},
            data=json.dumps(search_body),
            verify=False,
            timeout=10
        )
        
        if search_response.status_code == 200:
            results = search_response.json()
            hits = results['hits']['hits']
            
            print(f"🔍 Similarity Search Results for: '{query_text}'")
            print(f"Found {len(hits)} similar document chunks:\n")
            
            for i, hit in enumerate(hits, 1):
                score = hit['_score']
                source = hit['_source']
                
                print(f"Result {i} (Similarity Score: {score:.4f}):")
                print(f"  Document: {source.get('doc_name', 'N/A')}")
                print(f"  Chunk: {source.get('chunk_index', 'N/A')}")
                
                text_preview = source.get('text', '')[:200]
                print(f"  Content: {text_preview}...")
                print()
            
            return hits
        print(f"Search failed: {search_response.status_code}")
        print(f"Response: {search_response.text}")
        return None
            
    except requests.exceptions.ConnectionError:
        print("Cannot connect to OpenSearch or Ollama. Make sure both services are running.")
        return None
    except Exception as e:
        print(f"Error during similarity search: {e}")
        return None

# Example searches based on the ingested documents
print("=" * 80)
print("EXAMPLE 1: Search for information about docling-pipelines")
print("=" * 80)
perform_similarity_search("What is docling-pipelines and what does it do?", k=3)

print("\n" + "=" * 80)
print("EXAMPLE 2: Search for computing history")
print("=" * 80)
perform_similarity_search("Tell me about the history of computers and Alan Turing", k=3)

print("\n" + "=" * 80)
print("EXAMPLE 3: Search for document processing capabilities")
print("=" * 80)
perform_similarity_search("document extraction and embeddings", k=3)

## 4. Embedding Models Comparison

### Available Embedding Models

| Model | Dimensions | Provider | Speed | Quality |
|-------|-----------|----------|-------|----------|
| **nomic-embed-text** | 768 | Ollama | Fast | Very Good |
| **all-minilm** | 384 | Sentence Transformers | Fast | Good |
| **text-embedding-ada-002** | 1536 | OpenAI | Medium | Excellent |
| **granite-embedding** | 768 | IBM watsonx | Medium | Very Good |

### Choosing an Embedding Model

**For Local Development:**
- Use `nomic-embed-text` with Ollama (free, fast, good quality)

**For Production:**
- Use OpenAI `text-embedding-ada-002` (best quality, API costs)
- Use IBM watsonx `granite-embedding` (enterprise, good quality)

**For Resource-Constrained:**
- Use `all-minilm` (smaller dimensions, faster)

## 5. Vector Search Concepts

### Similarity Metrics

1. **L2 (Euclidean Distance)**
   - Measures straight-line distance
   - Good for general similarity
   - Used in this example

2. **Cosine Similarity**
   - Measures angle between vectors
   - Good for text similarity
   - Normalized by vector length

3. **Inner Product (Dot Product)**
   - Measures alignment
   - Fast computation
   - Sensitive to vector magnitude

### Search Algorithms

1. **HNSW (Hierarchical Navigable Small World)**
   - Fast approximate search
   - Good recall
   - Used in this example

2. **IVF (Inverted File Index)**
   - Partitions vector space
   - Good for large datasets
   - Faster but lower recall

3. **Flat (Brute Force)**
   - Exact search
   - Slow for large datasets
   - 100% recall

## 6. VectorDB Operator Concepts

### Understanding `available_features`

The `available_features` configuration tells the VectorDB operator which columns to store and how to handle them:

**Column Types:**
```python
"embeddings": {
    "type": "vector",              # Vector data (embeddings)
    "available_for_vector_db": True,  # Store in vector DB
    "mandatory_for_vector_db": True   # Required column
},
"content": {
    "type": "string",              # Text metadata
    "available_for_vector_db": True
}
```

**Why is this needed?**
- The VectorDB operator scans `available_features` to find columns with `type: "vector"`
- It uses `feature_mappings` to map your column names to OpenSearch field names
- Without this metadata, the operator cannot identify which columns contain embeddings

**Key Point:** Each operator in the pipeline adds features to `available_features`. The VectorDB operator uses this to understand what data is available and how to store it.

## Best Practices

### 1. Chunking Strategy
```python
# For semantic search
"chunk_size": 512,  # Good balance
"chunk_overlap": 50  # Preserve context

# For question answering
"chunk_size": 256,  # Smaller chunks
"chunk_overlap": 25
```

### 2. Embedding Generation
- Use batch processing for large datasets
- Cache embeddings to avoid regeneration
- Monitor API costs for cloud providers
- Use local models (Ollama) for development

### 3. Vector Database
- Choose appropriate index type (HNSW for speed, Flat for accuracy)
- Set proper vector dimensions
- Use batch inserts for better performance
- Monitor index size and query latency

### 4. Search Optimization
- Tune `k` parameter (number of results)
- Use filters to narrow search space
- Consider hybrid search (vector + keyword)
- Implement re-ranking for better results

## Summary

You've learned:
1. ✓ Generate embeddings with Ollama
2. ✓ Store vectors in OpenSearch
3. ✓ Perform similarity search on stored documents
4. ✓ Understand embedding models and their trade-offs
5. ✓ Learn vector search concepts (similarity metrics, algorithms)
6. ✓ Learn VectorDB operator concepts (`available_features`)
7. ✓ Best practices for embeddings and vector databases

## Next Steps

- **[05_quality_operators.ipynb](05_quality_operators.ipynb)** - Data quality assessment
- **[06_rag_pipeline.ipynb](06_rag_pipeline.ipynb)** - Complete RAG workflow
- **[07_flow_authoring.ipynb](07_flow_authoring.ipynb)** - Programmatic flow creation

## Learn More

- [Embeddings Operator Documentation](../../docs/operators/embeddings/embeddings_config.md)
- [VectorDB Operator Documentation](../../docs/operators/vectordb/vectordb_operator_config.md)
- [OpenSearch Documentation](../../docs/integrations/opensearch/README.md)
- [Ollama Documentation](https://ollama.ai/)